In [2]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder

In [3]:
data = pd.read_csv("./data_processed.csv")

In [4]:
data.head()

,Product Name,Category,Dosage Form,Price,Trademark,Brand Origin,Country,Rating,Continent,General_function
0,"Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...",Chăm sóc cơ thể,Gel,105000.0,DECUMAR,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
1,Dung dịch vệ sinh vùng kín Bimunica 250ml dành...,Chăm sóc cơ thể,Dạng kem,230000.0,Eucerin,Mỹ,Liên Bang Nga,5.0,Europe,Chăm sóc cơ thể
2,"Kem giảm thâm vùng nách, mông, bikini Neothera...",Chăm sóc cơ thể,Dạng kem,139000.0,La Beauty,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể
3,Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...,Chăm sóc cơ thể,Dạng kem,390000.0,SVR,Pháp,Pháp,unknown,Europe,Chăm sóc cơ thể
4,Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...,"Lăn khử mùi, xịt khử mùi",Dạng bọt,96000.0,Eucerin,Việt Nam,Việt Nam,5.0,Asia,Chăm sóc cơ thể


In [5]:
data.shape

(1999, 10)

In [6]:
# Create a label encoder object
label_encoder = preprocessing.LabelEncoder()

# Encode labels in the 'Country' column
data['Country'] = label_encoder.fit_transform(data['Country'])
data['Trademark'] = label_encoder.fit_transform(data['Trademark'])
data['General_function'] = label_encoder.fit_transform(data['General_function'])

print(data.head())

                                        Product Name  \
0  Gel chấm mụn, mờ thâm Decumar Advance THC 20g ...   
1  Dung dịch vệ sinh vùng kín Bimunica 250ml dành...   
2  Kem giảm thâm vùng nách, mông, bikini Neothera...   
3  Gel rửa mặt SVR Sebiaclear Gel Moussant 200ml ...   
4  Bọt vệ sinh nam giới Sumely Men's Sanitary Foa...   

                   Category Dosage Form     Price  Trademark Brand Origin  \
0           Chăm sóc cơ thể         Gel  105000.0         75     Việt Nam   
1           Chăm sóc cơ thể    Dạng kem  230000.0        118           Mỹ   
2           Chăm sóc cơ thể    Dạng kem  139000.0        217     Việt Nam   
3           Chăm sóc cơ thể    Dạng kem  390000.0        375         Pháp   
4  Lăn khử mùi, xịt khử mùi    Dạng bọt   96000.0        118     Việt Nam   

   Country   Rating Continent  General_function  
0       37      5.0      Asia                 0  
1       15      5.0    Europe                 0  
2       37      5.0      Asia                 0  


In [7]:
train_data = data[data['Rating'] != 'unknown']  # Sản phẩm có rating
prediction_data = data[data['Rating'] == 'unknown']  # Sản phẩm chưa có rating

In [8]:
train_data.shape, prediction_data.shape

((1220, 10), (779, 10))

In [9]:
train_data.to_csv("D:\\FILE_CSV_DATA_MODELING\\train_data.csv")

### 2. Feature Selection

In [10]:
features = ['Price', 'Trademark', 'Country', 'General_function']

### 3. Splitting dataset into X and y

In [11]:
X = train_data[features]
y = train_data["Rating"]
X_test = prediction_data[features]
y_test = prediction_data["Rating"]

In [12]:
X.head()

,Price,Trademark,Country,General_function
0,105000.0,75,37,0
1,230000.0,118,15,0
2,139000.0,217,37,0
4,96000.0,118,37,0
5,132000.0,118,10,1


In [13]:
y.head()

0    5.0
1    5.0
2    5.0
4    5.0
5    5.0
Name: Rating, dtype: object

##### X,y -> X_train, y_train, X_valid, y_valid

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state=0)

In [15]:
X.shape, X_train.shape, X_valid.shape

((1220, 4), (976, 4), (244, 4))

### **Model training**

Để lựa chọn thuật toán phù hợp với dữ liệu, chúng em tham khảo hướng dẫn từ tài liệu chính thức của Scikit-learn, bao gồm các tiêu chí sau:
1. Loại bài toán:
    * Đây là bài toán dự đoán giá trị liên tục (lượng đánh giá của khách hàng), do đó thuộc nhóm bài toán hồi quy (regression).
2. Quy mô dữ liệu:
    * Với kích thước dữ liệu hiện tại ít hơn 100K samples, chúng em lựa chọn các thuật toán phù hợp với dữ liệu nhỏ hoặc trung bình.
3. Thuật toán hồi quy được xem xét:

- Random Forest Regression: Mạnh mẽ trong việc xử lý dữ liệu phức tạp và không yêu cầu chuẩn hóa dữ liệu.
- Gradient Boosting (XGBoost): Hiệu quả cho các bài toán cần dự đoán chính xác, có khả năng xử lý outliers tốt.
- Ridge Regression: Phù hợp với dữ liệu nhỏ, thêm điều chuẩn để tránh overfitting.
- SVR(kernel=rbf): Tốt cho bài toán nhỏ và yêu cầu dự đoán chính xác, nhưng có thể chậm khi dữ liệu lớn.
- ElasticNet Regression: Kết hợp điều chuẩn L1 và L2, xử lý tốt với dữ liệu có nhiều đặc trưng không liên quan.

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,  make_scorer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

### **Random Forest algorithm**

In [17]:
#YOUR CODE HERE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

parameters = {
    'n_estimators' : [100, 200, 300, 400],
    'max_depth': [1, 2, 3, 4]
}
# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Huấn luyện mô hình
rf_regressor = RandomForestRegressor(random_state=0)
clf = GridSearchCV(rf_regressor, parameters)
clf.fit(X_train_scaled, y_train)

# Lấy giá trị tốt nhất của siêu tham số
best_params = clf.best_params_
print("Best Parameters:", best_params)

# Khởi tạo mô hình với các siêu tham số tốt nhất
rf_best = RandomForestRegressor(n_estimators=best_params['n_estimators'],
                                max_depth=best_params['max_depth'],
                                random_state=0)

# Huấn luyện lại mô hình với siêu tham số tốt nhất
rf_best.fit(X_train_scaled, y_train)

# Dự đoán với mô hình đã tối ưu
y_pred_best = rf_best.predict(X_valid_scaled)

y_valid = pd.to_numeric(y_valid, errors='coerce')

# Đánh giá mô hình
mae_best = mean_absolute_error(y_valid, y_pred_best)
mse_best = mean_squared_error(y_valid, y_pred_best)

mape = np.mean(np.abs((y_valid - y_pred_best) / y_valid)) * 100
accuracy_best = 100 - mape

print(f"MAE (Best Model): {mae_best}")
print(f"MSE (Best Model): {mse_best}")
print(f"Score (Best Model): {round(accuracy_best, 2)}%")

# Hiển thị kết quả
results_best = pd.DataFrame(zip(y_valid, y_pred_best, y_valid - y_pred_best), 
                            columns=['y_valid', 'y_pred', 'error'])

print(results_best.head(10))

Best Parameters: {'max_depth': 1, 'n_estimators': 100}
MAE (Best Model): 0.14269720342467876
MSE (Best Model): 0.06864955610689483
Score (Best Model): 96.76%
   y_valid    y_pred     error
0      5.0  4.892930  0.107070
1      5.0  4.917068  0.082932
2      5.0  4.897846  0.102154
3      5.0  4.898368  0.101632
4      5.0  4.917152  0.082848
5      5.0  4.916321  0.083679
6      5.0  4.915954  0.084046
7      5.0  4.916746  0.083254
8      5.0  4.890866  0.109134
9      5.0  4.893841  0.106159


### **Gradient Boosting algorithm**

In [18]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Hyperparameter grid for Gradient Boosting
gb_parameters = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [1, 2, 3, 4]
}

gb_regressor = GradientBoostingRegressor(random_state=0)
gb_clf = GridSearchCV(gb_regressor, gb_parameters, scoring='neg_mean_absolute_error', cv=3)
gb_clf.fit(X_train_scaled, y_train)

# Get best parameters and refit Gradient Boosting
best_gb_params = gb_clf.best_params_
print("Best Parameters (Gradient Boosting):", best_gb_params)
gb_best = GradientBoostingRegressor(n_estimators=best_gb_params['n_estimators'],
                                    learning_rate=best_gb_params['learning_rate'],
                                    max_depth=best_gb_params['max_depth'],
                                    random_state=0)
gb_best.fit(X_train_scaled, y_train)
gb_y_pred = gb_best.predict(X_valid_scaled)

# Evaluate Gradient Boosting model
gb_mae = mean_absolute_error(y_valid, gb_y_pred)
gb_mse = mean_squared_error(y_valid, gb_y_pred)
gb_mape = np.mean(np.abs((y_valid - gb_y_pred) / y_valid)) * 100
gb_accuracy = 100 - gb_mape

print(f"Gradient Boosting - MAE: {gb_mae}")
print(f"Gradient Boosting - MSE: {gb_mse}")
print(f"Gradient Boosting - Accuracy: {gb_accuracy:.2f}%")

# Display the results for Gradient Boosting
gb_results = pd.DataFrame(
    zip(y_valid, gb_y_pred, y_valid - gb_y_pred),
    columns=['y_valid', 'y_pred', 'error']
)

# Display the first 10 rows of results
gb_results.head(10)

Best Parameters (Gradient Boosting): {'learning_rate': 0.2, 'max_depth': 1, 'n_estimators': 300}
Gradient Boosting - MAE: 0.1574453586469206
Gradient Boosting - MSE: 0.08437480258653483
Gradient Boosting - Accuracy: 96.47%


,y_valid,y_pred,error
0,5.0,4.843083,0.156917
1,5.0,4.926635,0.073365
2,5.0,4.847690,0.152310
3,5.0,4.927102,0.072898
4,5.0,4.910937,0.089063
5,5.0,4.926635,0.073365
6,5.0,4.927956,0.072044
7,5.0,4.924143,0.075857
8,5.0,4.856381,0.143619
9,5.0,4.893602,0.106398


### **Ridge Regression algorithm**

In [19]:
#YOUR CODE HERE
from sklearn.preprocessing import PolynomialFeatures

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Tìm alpha tối ưu bằng GridSearchCV
params = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge_cv = GridSearchCV(Ridge(), params, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

# Lấy alpha tối ưu
best_alpha = ridge_cv.best_params_['alpha']
print("Best alpha:", best_alpha)

# Tăng khả năng học mối quan hệ phi tuyến
poly = PolynomialFeatures(degree=10)
X_train_poly = poly.fit_transform(X_train_scaled)
X_valid_poly = poly.transform(X_valid_scaled)

# Train lại mô hình với alpha tối ưu
rid_reg_poly = Ridge(alpha=best_alpha)
rid_reg_poly.fit(X_train_poly, y_train)
y_pred_poly = rid_reg_poly.predict(X_valid_poly)

# Đánh giá mô hình
r2 = r2_score(y_valid, y_pred_poly)
mae = mean_absolute_error(y_valid, y_pred_poly)
mse = mean_squared_error(y_valid, y_pred_poly)
rmse = np.sqrt(mse) 
mape = np.mean(np.abs((y_valid - y_pred_poly) / y_valid)) * 100
accuracy_best = 100 - mape

print("R^2 Score:", r2)
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print(f"Score (Best Model): {round(accuracy_best, 2)}%")
#pd.DataFrame({'y' : y_valid.head(), 'y_preds': y_pred_poly})

Best alpha: 100


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_ridge.py:254: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(


R^2 Score: -7.919524283042556
Mean Absolute Error (MAE): 0.1879566760238643
Mean Squared Error (MSE): 0.607339960785033
Root Mean Squared Error (RMSE): 0.7793201914393294
Score (Best Model): 95.86%


### **SVR(kernel=rbf)**

In [20]:
#YOUR CODE HERE
#YOUR CODE HERE
# Hàm chuẩn hóa dữ liệu
def preprocess_data(X_train, X_valid):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)
    return X_train_scaled, X_valid_scaled

# Hàm đánh giá mô hình
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"R² Score: {r2:.4f}")
    return mae, mse, r2

# 1. Chuẩn hóa dữ liệu
X_train_scaled, X_valid_scaled = preprocess_data(X_train, X_valid)

# 2. Khởi tạo và huấn luyện mô hình
kernel = 'rbf'  # Kernel cho SVR (có thể thử 'linear' hoặc 'poly')
C = 1.0         # Regularization parameter
epsilon = 0.1   # Độ lệch epsilon trong SVR

svr_model = SVR(kernel=kernel, C=C, epsilon=epsilon)
svr_model.fit(X_train_scaled, y_train)

# 3. Dự đoán và đánh giá mô hình
y_pred = svr_model.predict(X_valid_scaled)
evaluate_model(y_valid, y_pred)

mape = np.mean(np.abs((y_valid - y_pred) / y_valid)) * 100
accuracy_best = 100 - mape
print(f"Score (Best Model): {round(accuracy_best, 2)}%")


MAE: 0.1445
MSE: 0.0684
R² Score: -0.0041
Score (Best Model): 96.73%


### **ElasticNet**

In [21]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Tìm kiếm tham số alpha và l1_ratio tối ưu sử dụng GridSearchCV
param_grid = {
    'alpha': np.logspace(-4, 1, 10),  # Tìm kiếm alpha từ 0.0001 đến 10
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]  # Tìm kiếm l1_ratio từ 0 đến 1
}
elastic_net = ElasticNet(random_state=0)

# GridSearchCV để tìm các tham số tối ưu
grid_search = GridSearchCV(estimator=elastic_net, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=1)
grid_search.fit(X_train_scaled, y_train)

# Lấy các tham số tối ưu
best_alpha = grid_search.best_params_['alpha']
best_l1_ratio = grid_search.best_params_['l1_ratio']
print(f"Best alpha: {best_alpha}")
print(f"Best l1_ratio: {best_l1_ratio}")

# Train lại mô hình với alpha và l1_ratio tối ưu
elastic_net_best = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio, random_state=0)
elastic_net_best.fit(X_train_scaled, y_train)
y_pred = elastic_net_best.predict(X_valid_scaled)

# Đánh giá mô hình
r2 = r2_score(y_valid, y_pred)
mae = mean_absolute_error(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse) 
mape = np.mean(np.abs((y_valid - y_pred) / y_valid)) * 100
accuracy_best = 100 - mape

print("R^2 Score:", r2)
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print(f"Score (Best Model): {round(accuracy_best, 2)}%")


Best alpha: 0.05994842503189409
Best l1_ratio: 0.5
R^2 Score: -0.003609051350930814
Mean Absolute Error (MAE): 0.14345941951088378
Mean Squared Error (MSE): 0.06833681512026334
Root Mean Squared Error (RMSE): 0.261413111989937
Score (Best Model): 96.75%


### **Cross-validation** 

Khi so sánh các thuật toán mô hình hóa, việc tránh bias giữa các thuật toán là rất quan trọng. Để đánh giá các mô hình một cách khách quan và chính xác nhất, chúng ta sử dụng phương pháp Cross-Validation (K-fold), được cung cấp bởi Scikit-learn.

Phương pháp này hoạt động như sau:

- Chia tập dữ liệu huấn luyện thành K phần (folds) bằng nhau.
- Với mỗi lần lặp, giữ lại 1 fold để làm tập kiểm tra (validation set), và sử dụng K−1 folds còn lại để huấn luyện mô hình.
- Sau mỗi lần lặp, tính các metrics như RMSE (Root Mean Squared Error), MSE (Mean Squared Error).
- Sau khi hoàn thành K lần lặp, tổng hợp kết quả của từng fold và tính trung bình cộng (mean) của các metrics.
- Giá trị trung bình này được sử dụng để đánh giá hiệu suất mô hình một cách toàn diện và so sánh với các mô hình khác.

In [27]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
import time
import tracemalloc

# Cố định random seed để đảm bảo kết quả tái lập
seed = 2024

# Tạo mô hình Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],  # Số lượng cây trong Random Forest
    max_depth=best_params['max_depth'],       # Độ sâu tối đa của mỗi cây
    random_state=seed                         # Đảm bảo kết quả ổn định
)
rf_model.__class__.__name__ = "RandomForestRegression"  # Đổi tên để hiển thị dễ dàng hơn

# Tạo pipeline cho Ridge Regression với đặc trưng phi tuyến
ridge_poly_pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Chuẩn hóa dữ liệu
    ('poly', PolynomialFeatures(degree=2)),    # Tăng đặc trưng phi tuyến (bậc 2)
    ('ridge', Ridge(alpha=best_alpha))         # Ridge Regression với tham số điều chuẩn alpha
])  
ridge_poly_pipeline.__class__.__name__ = "RidgeRegression"  # Đổi tên để hiển thị

# Tạo Gradient Boosting Regressor
GradientBoosting_model = GradientBoostingRegressor(
    n_estimators=best_gb_params['n_estimators'],  # Số lượng cây
    learning_rate=best_gb_params['learning_rate'], # Tốc độ học
    max_depth=best_gb_params['max_depth'],         # Độ sâu tối đa
    random_state=seed                              # Đảm bảo kết quả ổn định
)
GradientBoosting_model.__class__.__name__ = "GradientBoostingRegression"

# Tạo ElasticNet Regressor
Elastic_model = ElasticNet(
    alpha=best_alpha,          # Tham số điều chuẩn L2
    l1_ratio=best_l1_ratio,    # Tỷ lệ giữa L1 và L2
    random_state=seed          # Đảm bảo kết quả ổn định
)
Elastic_model.__class__.__name__ = "ElasticNetRegression"

# Tạo Support Vector Regressor (SVR)
svr_model = SVR(
    kernel=kernel,   # Kernel sử dụng (ví dụ: 'rbf')
    C=C,             # Tham số điều chuẩn
    epsilon=epsilon  # Biên sai số
)
svr_model.__class__.__name__ = "SVR(kernel=rbf)"

# Danh sách các mô hình cần so sánh
models = [
    rf_model,
    ridge_poly_pipeline,
    Elastic_model,
    GradientBoosting_model,
    svr_model
]

# Hàm tính RMSE (Root Mean Square Error)
rmse_scorer = make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=False)

# Hàm tính MSE (Mean Squared Error)
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# Hàm đánh giá các mô hình
def generate_baseline_results(models, X, y, cv=5):
    kfold = KFold(n_splits=cv, shuffle=True, random_state=seed)  # K-Fold Cross Validation
    entries = []  # Lưu kết quả của mỗi mô hình

    for model in models:
        model_name = model.__class__.__name__  # Tên của mô hình
        rmse_scores = []  # Lưu các giá trị RMSE của từng fold
        mse_scores = []   # Lưu các giá trị MSE của từng fold
        times = []        # Lưu thời gian huấn luyện
        memory_usages = []  # Lưu bộ nhớ tiêu thụ

        for train_idx, valid_idx in kfold.split(X, y):
            # Chia dữ liệu thành tập train và validation
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            # Bắt đầu theo dõi thời gian và bộ nhớ
            start_time = time.time()  # Ghi lại thời gian bắt đầu
            tracemalloc.start()      # Bắt đầu theo dõi bộ nhớ

            # Huấn luyện mô hình
            model.fit(X_train, y_train)

            # Dự đoán trên tập validation
            y_pred = model.predict(X_valid)

            # Tính toán các chỉ số đánh giá
            rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
            mse = mean_squared_error(y_valid, y_pred)

            # Lấy thông tin về bộ nhớ tiêu thụ
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            # Tính thời gian huấn luyện
            elapsed_time = time.time() - start_time

            # Lưu lại các kết quả
            rmse_scores.append(rmse)
            mse_scores.append(mse)
            times.append(elapsed_time)
            memory_usages.append(peak / 1024 / 1024)  # Đổi sang MB

        # Tính trung bình các kết quả
        entries.append({
            'model_name': model_name,
            'mean_rmse': np.mean(rmse_scores),          # RMSE trung bình
            'mean_mse': np.mean(mse_scores),            # MSE trung bình
            'mean_time': np.mean(times),                # Thời gian huấn luyện trung bình
            'mean_memory_usage': np.mean(memory_usages) # Bộ nhớ tiêu thụ trung bình (MB)
        })

    # Chuyển kết quả thành DataFrame
    baseline_results = pd.DataFrame(entries)
    baseline_results.sort_values(by=['mean_rmse'], ascending=True, inplace=True)  # Sắp xếp theo RMSE

    return baseline_results

# Gọi hàm để đánh giá các mô hình
generate_baseline_results(models, X, y, cv=5)

,model_name,mean_rmse,mean_mse,mean_time,mean_memory_usage
4,SVR(kernel=rbf),0.340784,0.130763,0.012831,0.101313
2,ElasticNetRegression,0.341830,0.131375,0.006788,0.107529
0,RandomForestRegression,0.342873,0.132255,0.547997,0.191311
1,RidgeRegression,0.342948,0.132240,0.009428,0.311690
3,GradientBoostingRegression,0.349008,0.135000,0.397760,0.404256


### **Choose the best model and predict**

- Mô hình tốt nhất được chọn dựa trên giá trị Mean RMSE và MSE thấp nhất từ kết quả Cross-Validation.
- Sau khi chọn được mô hình tốt nhất, tiến hành huấn luyện lại mô hình trên toàn bộ tập dữ liệu để tăng độ chính xác, sau đó dự đoán **Rating** cho các sản phẩm chưa có thông tin và lưu kết quả vào cột mới **Predicted Rating**
    

In [29]:

baseline_results = generate_baseline_results(models, X, y, cv=5)

# Chọn mô hình tốt nhất
best_model_name = baseline_results.iloc[0]['model_name']  # Lấy tên mô hình có Mean RMSE nhỏ nhất
print(f"\nBest Model Selected: {best_model_name}")

if best_model_name == "Ridge":
    # Nếu mô hình tốt nhất là Ridge 
    best_model = ridge_poly_pipeline
else:
    # Nếu mô hình tốt nhất là RandomForestRegressor
    best_model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=seed)
    
# Huấn luyện lại mô hình tốt nhất trên toàn bộ tập train
best_model.fit(X, y)

# Dự đoán rating cho tập prediction_data
predictions = best_model.predict(X_test)

# Xuất kết quả dự đoán
prediction_data['Predicted_Rating'] = predictions
print("\nPredicted Ratings for 'unknown':")
print(prediction_data[['Price', 'Trademark', 'Country', 'General_function', 'Predicted_Rating']])

# Lưu kết quả ra file 
prediction_data.to_csv("predicted_ratings.csv", index=False)


Best Model Selected: SVR(kernel=rbf)

Predicted Ratings for 'unknown':
         Price  Trademark  Country  General_function  Predicted_Rating
3     390000.0        375       23                 0          4.942197
10     22400.0        214       37                 3          4.915070
14    151200.0        290       37                 0          4.966961
30     39000.0         14       37                 0          4.973266
31     39000.0         14       37                 0          4.973266
...        ...        ...      ...               ...               ...
1994  276000.0        208       37                14          4.975028
1995  276000.0        342       37                 4          4.772101
1996  276000.0        342       37                15          4.498044
1997  276000.0        234       18                 5          4.938452
1998  276000.0        157        7                15          4.837847

[779 rows x 5 columns]


C:\Users\Admin\AppData\Local\Temp\ipykernel_15336\2678503895.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prediction_data['Predicted_Rating'] = predictions
